In [28]:
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from gen_variable_standard_static import metrics_search_for_fragment_df, \
metrics_search_for_two_fragments_df, widened_search_for_fragment_df, \
widened_search_for_two_fragments_df, metadata_agg_name, \
metadata_agg_subtype_name, sources_checker, \
find_aggregate_variable_names_gen_mod, \
metadata_part_name, metadata_part_subtype_name, \
find_all_variable_names_gen_mod

In [29]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')

In [30]:
all_data_systems = systems_cleaned[
    systems_cleaned['has_current_data']
    & systems_cleaned['has_irradiance_data']
    & systems_cleaned['has_power_data']
    & systems_cleaned['has_temperature_data']
    & systems_cleaned['has_voltage_data']
]
all_data_ids = set(all_data_systems.system_id)

In [31]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(
    metrics_dir,
)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [32]:
my_system_ids = list(all_data_ids.intersection(metrics_id_set))
my_system_ids.sort()
num_ids = len(my_system_ids)
num_ids

39

## DC Voltage Starter

In [33]:
j = 0
system_id = my_system_ids[j]
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
relevant_dc_volts = metrics_search_for_two_fragments_df(
    relevant_rows_metrics, 'dc', 'volt', 'and'
)
relevant_dc_volts

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
1299,2,351,das_battery_voltage,DC voltage battery,V,V,1.0,0.0,,avg,NaN,NaN,,das_battery_voltage__351
1302,2,347,dc_pos_voltage,DC voltage,V,V,1.0,0.0,,avg,NaN,NaN,,dc_pos_voltage__347


Note: As from the first-few rich-data systems, often a `dc_pos_voltage` without a `dc_neg_voltage`, just a `das_battery_voltage`.  This makes the agg-vs-parts work very interesting, as System 50 has pos and neg, and no real combination.

*Many* systems have subparts without an aggregate (1200, 1305, 1306,1307,1308,1310, 1403...)  -- more without aggregate than with, I think!

In [34]:
dc_volts_agg_names = ['dc_voltage', 'input_voltage', 'dc_voltage_Vdc',
                      'InvVDCin_Avg', 'InvPVin_Avg',]  # dc_pos_voltage and dc_neg_voltage removed for now, as 'parts' in a sense
dc_volts_bat_agg_names = ['das_battery_voltage', 'Battery_V_Min']

In [35]:
def find_aggregate_dc_voltage(
    systems_cleaned: pd.DataFrame,
    metrics_df: pd.DataFrame,
    print_messages: bool,
    include_battery: bool,
    known_sources=('inverter', 'battery'),
    known_sources_short=('inv', 'batt')
):
    filtered_metrics_df = metrics_search_for_two_fragments_df(
        metrics_df, 'dc', 'volt', 'and'
    )
    if not include_battery:
        battery_df = metrics_search_for_fragment_df(
            metrics_df, 'batt'
        )
        filtered_metrics_df = filtered_metrics_df.drop(
            index = battery_df.index
        )
    dc_volts_agg_names = ['dc_voltage', 'input_voltage', 'dc_voltage_Vdc',
                          'InvVDCin_Avg', 'InvVPVin_Avg',]
    dc_volts_bat_agg_names = ['das_battery_voltage', 'Battery_V_Min']
    all_dc_volts = dc_volts_agg_names + dc_volts_bat_agg_names
    if include_battery:
        return find_aggregate_variable_names_gen_mod(
            systems_cleaned=systems_cleaned,
            filtered_metrics_df=filtered_metrics_df,
            var_name='dc_voltage',
            agg_var_sensor_names=all_dc_volts,
            print_messages=print_messages,
            sources_matter=True,
            known_sources=known_sources,
            known_sources_short=known_sources_short
        )
    else:
        return find_aggregate_variable_names_gen_mod(
            systems_cleaned=systems_cleaned,
            filtered_metrics_df=filtered_metrics_df,
            var_name='dc_voltage',
            agg_var_sensor_names=dc_volts_agg_names,
            print_messages=print_messages,
            sources_matter=False,
            known_sources=known_sources,
            known_sources_short=known_sources_short
        )


In [36]:
all_agg_volts, all_aggvolt_metadata = find_aggregate_dc_voltage(
    systems_cleaned=systems_cleaned,
    metrics_df=metrics_df,
    print_messages=True,
    include_battery=True
)

System 1283 appears to have no obvious dc_voltage aggregator name.
System 1419 appears to have no obvious dc_voltage aggregator name.
System 1422 appears to have no obvious dc_voltage aggregator name.
System 1423 appears to have no obvious dc_voltage aggregator name.
System 1429 appears to have no obvious dc_voltage aggregator name.
System 1305 appears to have no obvious dc_voltage aggregator name.
System 1306 appears to have no obvious dc_voltage aggregator name.
System 1307 appears to have no obvious dc_voltage aggregator name.
System 1308 appears to have no obvious dc_voltage aggregator name.
System 1310 appears to have no obvious dc_voltage aggregator name.
System 1312 appears to have no obvious dc_voltage aggregator name.
System 4901 has multiple dc_voltage aggregators from the inverter source:
{'metric_id': np.int32(82506), 'sensor_name': 'InvVDCin_Avg', 'common_name': 'DC voltage other', 'units': 'V', 'calc_details': '', 'whole_or_part': 'whole', 'source_type': 'inverter'}
{'met

In [37]:
def find_all_dc_voltage(
    systems_cleaned: pd.DataFrame,
    metrics_df: pd.DataFrame,
    print_messages: bool,
    include_battery: bool,
    known_sources=('inverter', 'battery'),
    known_sources_short=('inv', 'batt')
):
    filtered_metrics_df = metrics_search_for_two_fragments_df(
        metrics_df, 'dc', 'volt', 'and'
    )
    if not include_battery:
        battery_df = metrics_search_for_fragment_df(
            metrics_df, 'batt'
        )
        filtered_metrics_df = filtered_metrics_df.drop(
            index = battery_df.index
        )
    dc_volts_agg_names = ['dc_voltage', 'input_voltage', 'dc_voltage_Vdc',
                          'InvVDCin_Avg', 'InvVPVin_Avg',]
    dc_volts_bat_agg_names = ['das_battery_voltage', 'Battery_V_Min']
    all_dc_volts = dc_volts_agg_names + dc_volts_bat_agg_names
    all_agg_volts, all_aggvolt_metadata = find_aggregate_dc_voltage(
        systems_cleaned=systems_cleaned,
        metrics_df=metrics_df,
        print_messages=print_messages,
        include_battery=include_battery,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )
    # Requires the modified function from the updated
    # gen_variable_standard_static.py
    if include_battery:
        all_ac_volts, all_ac_volts_metadata = find_all_variable_names_gen_mod(
            var_aggs_dict=all_agg_volts,
            var_aggs_metadata=all_aggvolt_metadata,
            filtered_metrics_df=filtered_metrics_df,
            var_name='dc_voltage',
            agg_var_sensor_names=all_dc_volts,
            sources_matter=True,
            known_sources=known_sources,
            known_sources_short=known_sources_short,
            hard_stop_on_singleton=False
        )
    else:
        all_ac_volts, all_ac_volts_metadata = find_all_variable_names_gen_mod(
            var_aggs_dict=all_agg_volts,
            var_aggs_metadata=all_aggvolt_metadata,
            filtered_metrics_df=filtered_metrics_df,
            var_name='dc_voltage',
            agg_var_sensor_names=dc_volts_agg_names,
            sources_matter=False,
            known_sources=known_sources,
            known_sources_short=known_sources_short,
            hard_stop_on_singleton=False
        )
    # cleanup -- if solo 'positive' part-voltage (no negative),
    # go ahead and upgrade it to an aggregate
    for system_id in all_ac_volts.keys():
        num_parts = 0
        num_pos_parts = 0
        num_neg_parts = 0
        for metric_dict in all_ac_volts[system_id]:
            if metric_dict['whole_or_part'] == 'part':
                num_parts += 1
                if 'pos' in metric_dict['sensor_name']:
                    num_pos_parts += 1
                elif 'neg' in metric_dict['sensor_name']:
                    num_neg_parts += 1
        if (num_neg_parts == 0) and (num_pos_parts == 1):
            # unique positive part upgraded to an aggregate
            all_ac_volts_metadata.loc[
                system_id, metadata_agg_name(var_name='dc_voltage')
            ] = True
            if num_parts == 1:
                all_ac_volts_metadata.loc[
                    system_id, metadata_part_name(var_name='dc_voltage')
                ] = False
            for metric_dict in all_ac_volts[system_id]:
                if (
                    (metric_dict['whole_or_part'] == 'part')
                    and ('pos' in metric_dict['sensor_name'])
                ):
                    metric_dict['whole_or_part'] = 'whole'
                    metric_dict.pop('index')
                    if 'source' in metric_dict:
                        all_ac_volts_metadata.loc[
                            system_id, metadata_agg_subtype_name(
                                'dc_voltage', metric_dict['source_type']
                            )
                        ] = True
                        if num_parts == 1:
                            all_ac_volts_metadata.loc[
                                system_id, 
                                metadata_part_subtype_name(
                                    'dc_voltage', metric_dict['source_type']
                                )
                            ] = False
                    # only one positive part by hypothesis, so
                    break
        elif num_neg_parts < num_pos_parts:
            print(f'For System {system_id}, an awkward mismatch between parts.\n'
                  + f'Positive parts: {num_pos_parts}, Negative parts: {num_neg_parts}.\n'
                  + 'Check and see if all of the output looks right.')
    return (all_ac_volts, all_ac_volts_metadata)
    

In [38]:
all_volts, all_metadata = find_all_dc_voltage(
    systems_cleaned=systems_cleaned,
    metrics_df=metrics_df,
    print_messages=False,
    include_battery=True
)

System 2 has only one subpart for dc_voltage!
system_id                             2
metric_id                           347
sensor_name              dc_pos_voltage
common_name                  DC voltage
raw_units                             V
units                                 V
calc_scale                          1.0
calc_offset                         0.0
calc_details                           
aggregation_type                    avg
source_type                         NaN
source_id                           NaN
comments                               
standard_name       dc_pos_voltage__347
Name: 1302, dtype: object
0
0
1
System 3 has only one subpart for dc_voltage!
system_id                             3
metric_id                           354
sensor_name              dc_pos_voltage
common_name                  DC voltage
raw_units                             V
units                                 V
calc_scale                          1.0
calc_offset                         

In [39]:
all_volts[2]

[{'metric_id': np.int32(351),
  'sensor_name': 'das_battery_voltage',
  'common_name': 'DC voltage battery',
  'units': 'V',
  'calc_details': '',
  'whole_or_part': 'whole',
  'source_type': 'battery'},
 {'metric_id': np.int32(347),
  'sensor_name': 'dc_pos_voltage',
  'common_name': 'DC voltage',
  'units': 'V',
  'calc_details': '',
  'whole_or_part': 'whole',
  'source_type': 'unknown'}]

In [40]:
part_metric_names = []
for system_id in all_volts.keys():
    for metric_dict in all_volts[system_id]:
        if metric_dict['whole_or_part'] == 'part':
            part_metric_names.append(metric_dict['sensor_name'])
part_metric_names.sort()
for metric_name in part_metric_names:
    print(metric_name)

dc_neg_voltage
dc_neg_voltage
dc_pos_voltage
dc_pos_voltage
dc_voltage_1
dc_voltage_1
dc_voltage_1
dc_voltage_1
dc_voltage_1
dc_voltage_1
dc_voltage_1
dc_voltage_1_Vdc
dc_voltage_2
dc_voltage_2
dc_voltage_2
dc_voltage_2
dc_voltage_2
dc_voltage_2
dc_voltage_2
dc_voltage_2_Vdc
dc_voltage_3
dc_voltage_3
dc_voltage_3
dc_voltage_3
dc_voltage_3
dc_voltage_3
dc_voltage_3
dc_voltage_4
dc_voltage_4
dc_voltage_5
dc_voltage_5
dc_voltage_6
dc_voltage_6
dc_voltage_A
dc_voltage_B
dc_voltage_C
dc_voltage_negative
dc_voltage_positive
inv1_dc_voltage
inv1_dc_voltage
inv1_dc_voltage
inv1_dc_voltage
inv1_dc_voltage
inv1_dc_voltage
inv1_dc_voltage
inv1_dc_voltage
inv1_dc_voltage
inv1_dc_voltage
inv1_dc_voltage
inv1_dc_voltage
inv1_dc_voltage
inv1_input_voltage
inv2_dc_voltage
inv2_dc_voltage
inv2_dc_voltage
inv2_dc_voltage
inv2_dc_voltage
inv2_dc_voltage
inv2_dc_voltage
inv2_dc_voltage
inv2_dc_voltage
inv2_dc_voltage
inv2_dc_voltage
inv2_dc_voltage
inv2_dc_voltage
inv2_input_voltage
inv3_dc_voltage
inv3_d

Ok, so we dropped the errors, and 'promoted' dc_pos_voltage to aggregate successfully.  But still need to resolve 4901-4903 issues of which term is important to avoid name-repeats.

## AC Voltage Starter

In [41]:
# j = 15
# system_id = my_system_ids[j]
system_id = 1214
relevant_rows_metrics = metrics_df[metrics_df['system_id'] == system_id]
relevant_ac_volts = metrics_search_for_two_fragments_df(
    relevant_rows_metrics, 'ac', 'volt', 'and'
)
relevant_ac_volts

,system_id,metric_id,sensor_name,common_name,raw_units,units,calc_scale,calc_offset,calc_details,aggregation_type,source_type,source_id,comments,standard_name
240,1214,3184,PPV_avg,AC voltage,V,V,1.0,0.0,,avg,NaN,NaN,,ppv_avg__3184


Immediate observation: battery or no battery, sources matter!
Still lots of missingness for aggregates!

In [42]:
ac_voltage_agg_names = [
    'InvVabcAvg_Avg', 'ac_voltage_Vac',
    'ac_voltage', 'ac_volts',
]

In [43]:
def find_aggregate_ac_voltage(
    systems_cleaned: pd.DataFrame,
    metrics_df: pd.DataFrame,
    print_messages: bool,
    known_sources=('inverter', 'meter'),
    known_sources_short=('inv', 'met')
):
    filtered_metrics_df = metrics_search_for_two_fragments_df(
        metrics_df, 'ac', 'volt', 'and'
    )
    ac_voltage_agg_names = [
        'InvVabcAvg_Avg', 'ac_voltage_Vac',
        'ac_voltage', 'ac_volts', 'PPV_avg'
    ]
    return find_aggregate_variable_names_gen_mod(
        systems_cleaned=systems_cleaned,
        filtered_metrics_df=filtered_metrics_df,
        var_name='dc_voltage',
        agg_var_sensor_names=ac_voltage_agg_names,
        print_messages=print_messages,
        sources_matter=True,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )


In [44]:
all_agg_ac_voltage, all_agg_ac_voltage_metadata = find_aggregate_ac_voltage(
    systems_cleaned=systems_cleaned,
    metrics_df=metrics_df,
    print_messages=True
)

System 1416 appears to have no obvious dc_voltage aggregator name.
System 1422 appears to have no obvious dc_voltage aggregator name.
System 1423 appears to have no obvious dc_voltage aggregator name.
System 1429 appears to have no obvious dc_voltage aggregator name.
System 1305 appears to have no obvious dc_voltage aggregator name.
System 1306 appears to have no obvious dc_voltage aggregator name.
System 1307 appears to have no obvious dc_voltage aggregator name.
System 1308 appears to have no obvious dc_voltage aggregator name.
System 1310 appears to have no obvious dc_voltage aggregator name.
System 1312 appears to have no obvious dc_voltage aggregator name.
System 4901 appears to have no obvious dc_voltage aggregator name.
System 4902 appears to have no obvious dc_voltage aggregator name.
System 1203 appears to have no obvious dc_voltage aggregator name.
System 1368 appears to have no obvious dc_voltage aggregator name.
System 1369 appears to have no obvious dc_voltage aggregator n

In [45]:
def find_all_ac_voltage(
    systems_cleaned: pd.DataFrame,
    metrics_df: pd.DataFrame,
    print_messages: bool,
    known_sources=('inverter', 'meter'),
    known_sources_short=('inv', 'met')
):
    filtered_metrics_df = metrics_search_for_two_fragments_df(
        metrics_df, 'ac', 'volt', 'and'
    )
    # no percentages
    filtered_metrics_df = filtered_metrics_df[filtered_metrics_df['units'] == 'V']
    ac_voltage_agg_names = [
        'InvVabcAvg_Avg', 'ac_voltage_Vac',
        'ac_voltage', 'ac_volts', 'PPV_avg'
    ]
    all_agg_ac_volts, all_agg_ac_volt_metadata = find_aggregate_ac_voltage(
        systems_cleaned=systems_cleaned,
        metrics_df=metrics_df,
        print_messages=print_messages,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )
    return find_all_variable_names_gen_mod(
        var_aggs_dict=all_agg_ac_volts,
        var_aggs_metadata=all_agg_ac_volt_metadata,
        filtered_metrics_df=filtered_metrics_df,
        var_name='ac_voltage',
        agg_var_sensor_names=ac_voltage_agg_names,
        sources_matter=True,
        known_sources=known_sources,
        known_sources_short=known_sources_short
    )


In [46]:
all_ac_volts, all_ac_metadata = find_all_ac_voltage(
    systems_cleaned=systems_cleaned,
    metrics_df=metrics_df,
    print_messages=False,
)

In [48]:
all_ac_volts

{1284: [{'metric_id': np.int32(947),
   'sensor_name': 'ac_voltage',
   'common_name': 'AC voltage',
   'units': 'V',
   'calc_details': '',
   'whole_or_part': 'whole',
   'source_type': 'unknown'}],
 4: [{'metric_id': np.int32(318),
   'sensor_name': 'ac_voltage',
   'common_name': 'AC voltage',
   'units': 'V',
   'calc_details': '',
   'whole_or_part': 'whole',
   'source_type': 'unknown'}],
 1416: [{'metric_id': np.int32(4756),
   'sensor_name': 'ac_voltage_AB',
   'common_name': 'AC voltage',
   'units': 'V',
   'calc_details': '',
   'whole_or_part': 'part',
   'source_type': 'unknown',
   'index': 'AB'},
  {'metric_id': np.int32(4758),
   'sensor_name': 'ac_voltage_AC',
   'common_name': 'AC voltage',
   'units': 'V',
   'calc_details': '',
   'whole_or_part': 'part',
   'source_type': 'unknown',
   'index': 'AC'},
  {'metric_id': np.int32(4759),
   'sensor_name': 'ac_voltage_AN',
   'common_name': 'AC voltage',
   'units': 'V',
   'calc_details': '',
   'whole_or_part': 'part'

In [20]:
units_list = []
for system_id in all_ac_volts.keys():
    for metric_dict in all_ac_volts[system_id]:
        units_list.append(metric_dict['units'])
units_list = set(units_list)
units_list
    

{'V'}

In [47]:
part_metric_names = []
for system_id in all_ac_volts.keys():
    for metric_dict in all_ac_volts[system_id]:
        if metric_dict['whole_or_part'] == 'part':
            part_metric_names.append(metric_dict['sensor_name'])
part_metric_names.sort()
for metric_name in part_metric_names:
    print(metric_name)

InvVa_Avg
InvVa_Avg
InvVa_Avg
InvVb_Avg
InvVb_Avg
InvVb_Avg
InvVc_Avg
InvVc_Avg
InvVc_Avg
PwrMtrVa_Avg
PwrMtrVa_Avg
PwrMtrVa_Avg
PwrMtrVb_Avg
PwrMtrVb_Avg
PwrMtrVb_Avg
PwrMtrVc_Avg
PwrMtrVc_Avg
PwrMtrVc_Avg
ac_voltage_1
ac_voltage_1
ac_voltage_1
ac_voltage_1
ac_voltage_1
ac_voltage_1
ac_voltage_2
ac_voltage_2
ac_voltage_2
ac_voltage_2
ac_voltage_2
ac_voltage_2
ac_voltage_3
ac_voltage_3
ac_voltage_3
ac_voltage_3
ac_voltage_3
ac_voltage_3
ac_voltage_4
ac_voltage_4
ac_voltage_5
ac_voltage_5
ac_voltage_6
ac_voltage_6
ac_voltage_AB
ac_voltage_AC
ac_voltage_AN
ac_voltage_BC
ac_voltage_BN
ac_voltage_CN
ac_voltage_LtoL
ac_voltage_LtoN
inv1_ac_voltage
inv1_ac_voltage
inv1_ac_voltage
inv1_ac_voltage
inv1_ac_voltage
inv1_ac_voltage
inv1_ac_voltage
inv1_ac_voltage
inv2-9_ac_voltage
inv2_ac_voltage
inv2_ac_voltage
inv2_ac_voltage
inv2_ac_voltage
inv2_ac_voltage
inv2_ac_voltage
inv2_ac_voltage
inv3_ac_voltage
inv4_ac_voltage
inv5_ac_voltage
inv6_ac_voltage


All good!